# 24 — Post processing

Helpers that decorate atlas outputs with country context so downstream analyses can group / filter by country or continent. Also adds raw data of _apparent_temp_monthly.nc, _precipitation_monthly.nc and _population_density.nc

In [ ]:
import geopandas as gpd
import httpx
import pandas as pd
import xarray as xr

from common import PROCESSED_DIR, RAW_DIR

variable_raw = RAW_DIR / 'post_processing'
variable_raw.mkdir(parents=True, exist_ok=True)

## 1. Country polygons

Natural Earth 50m admin_0 boundaries — same shapefile used by `18_natural_disaster_risk` and `19_climate_vulnerability`. Loaded here directly (not via `regionmask`) for the `ISO_A3_EH` and `CONTINENT` attributes.

In [ ]:
NE_URL = 'https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_0_countries.zip'
ne_zip = variable_raw / 'ne_50m_admin_0_countries.zip'

if not ne_zip.exists():
    print(f'downloading {NE_URL}')
    r = httpx.get(NE_URL, headers={'User-Agent': 'Mozilla/5.0'}, timeout=180, follow_redirects=True)
    r.raise_for_status()
    ne_zip.write_bytes(r.content)

print(f'{ne_zip.name}: {ne_zip.stat().st_size / 1024**2:.1f} MB')

In [ ]:
countries = (
    gpd.read_file(f'zip://{ne_zip}')
    .to_crs('EPSG:4326')
    [['ISO_A3_EH', 'CONTINENT', 'geometry']]
    .rename(columns={'ISO_A3_EH': 'country_code', 'CONTINENT': 'continent'})
)
countries.head()

## 2. Enrich a dataframe

Point-in-polygon join against the Natural Earth polygons. Rows whose point falls in the ocean or in a disputed area not covered by any polygon get NaN for both columns.

In [ ]:
def add_country_and_continent(df, lat_col='lat', lon_col='lon', polygons=None):
    """Enrich ``df`` with ``country_code`` (ISO3) and ``continent`` columns.

    Each row's ``(lat, lon)`` is looked up against Natural Earth 50m admin_0
    country polygons via a spatial join. Rows whose point falls outside every
    polygon (ocean cells, unclaimed territory) get NaN in both columns.

    Parameters
    ----------
    df : pandas.DataFrame
        Input frame with latitude and longitude columns in EPSG:4326.
    lat_col, lon_col : str
        Column names holding the coordinates.
    polygons : geopandas.GeoDataFrame, optional
        Country polygons with ``country_code`` and ``continent`` columns.
        Defaults to the ``countries`` frame loaded above.

    Returns
    -------
    pandas.DataFrame
        Copy of ``df`` with two extra columns appended. Row order preserved.
    """
    polys = countries if polygons is None else polygons
    points = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs='EPSG:4326',
    )
    joined = points.sjoin(
        polys[['country_code', 'continent', 'geometry']],
        how='left',
        predicate='within',
    )
    # sjoin can duplicate rows when a point sits exactly on a shared border;
    # keep the first hit so output length matches input length.
    joined = joined[~joined.index.duplicated(keep='first')]
    return pd.DataFrame(joined.drop(columns=['geometry', 'index_right']))

## 3. Enrich the normalized atlas

Load `normalized.nc` (all percentile-transformed layers from `92_normalization`), flatten the `(lat, lon)` grid into rows, attach `country_code` and `continent`, and save as `normalized_by_country.parquet` for downstream country / continent aggregations.

In [ ]:
ds = xr.open_dataset(PROCESSED_DIR / 'normalized.nc')
df = ds.to_dataframe().reset_index()

# Cells that are NaN in every layer are pure ocean or coverage gaps — they
# have nothing to aggregate later, so drop before the (expensive) spatial join.
layer_cols = [c for c in df.columns if c not in ('lat', 'lon')]
df = df.dropna(subset=layer_cols, how='all').reset_index(drop=True)

df = add_country_and_continent(df)

n_missing = df['country_code'].isna().sum()
print(f'{len(df):,} cells; {n_missing:,} ({n_missing / len(df):.1%}) with no country match')

out = PROCESSED_DIR / 'normalized_by_country.parquet'
df.to_parquet(out, index=False)
print(f'wrote {out}')
df.head()